# FraudGraph - Phase 1: EDA, Strict Time Split & XGBoost Baselines

**Goal:** Know your data and establish benchmark baselines that the Hybrid GNN has to beat.
- **Dataset:** IEEE-CIS Fraud Detection (`train_transaction.csv`, `train_identity.csv`)
- **Split:** Strictly chronological (70% Train, 10% Validation, 20% Test)
- **Baselines:**
  - **Baseline A:** Weak XGBoost (8 engineered features, target AUC ~0.769)
  - **Baseline B:** Full Tabular XGBoost (~430 features, target AUC ~0.907)
- **Evaluation Metrics:** ROC-AUC, Average Precision (PR-AUC), and Precision at 10% Recall


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import mlflow
from pathlib import Path

# Path setup (Kaggle vs local)
if os.path.exists('/kaggle/input/ieee-fraud-detection'):
    DATA_DIR = Path('/kaggle/input/ieee-fraud-detection')
    ARTIFACTS_DIR = Path('/kaggle/working/artifacts')
else:
    DATA_DIR = Path('../data')
    ARTIFACTS_DIR = Path('../artifacts')

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data Directory: {DATA_DIR}')
print(f'Artifacts Directory: {ARTIFACTS_DIR}')


## 1. Load Data and Merge Tables
Join transaction and identity tables on `TransactionID`.

In [ ]:
print('Loading transaction table...')
df_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
id_file = DATA_DIR / 'train_identity.csv'
if id_file.exists():
    print('Loading identity table...')
    df_id = pd.read_csv(id_file)
    df = df_tx.merge(df_id, on='TransactionID', how='left')
else:
    df = df_tx
print(f'Merged dataset shape: {df.shape}')


## 2. Quick Exploratory Data Analysis (EDA)
- Fraud rate: ~3.5%
- Device info availability: ~20%
- Missing values across columns

In [ ]:
total_tx = len(df)
fraud_rate = df['isFraud'].mean()
print(f'Total transactions: {total_tx:,}')
print(f'Base fraud rate: {fraud_rate:.4%} (~3.5% expected)')

device_cols = [c for c in ['DeviceInfo', 'DeviceType'] if c in df.columns]
if device_cols:
    has_device = (~df[device_cols[0]].isna() | ~df[device_cols[-1]].isna()).mean()
    print(f'Device info coverage: {has_device:.4%} (~20% expected)')

print('Transaction Amount Summary:')
print(df['TransactionAmt'].describe())


## 3. Strict Time-Based Split
**Crucial Leakage Prevention:** Fraud data is time-ordered. Random folds leak future knowledge into past predictions.
- 70% Train
- 10% Validation (hyperparameters, early stopping)
- 20% Test (scored strictly once per model)

In [ ]:
df_sorted = df.sort_values(by='TransactionDT').reset_index(drop=True)
n = len(df_sorted)
n_train = int(n * 0.70)
n_val = int(n * 0.80)

train_df = df_sorted.iloc[:n_train]
val_df = df_sorted.iloc[n_train:n_val]
test_df = df_sorted.iloc[n_val:]

boundaries = {
    't_train_start': int(train_df['TransactionDT'].iloc[0]),
    't_val_start': int(val_df['TransactionDT'].iloc[0]),
    't_test_start': int(test_df['TransactionDT'].iloc[0]),
    't_max': int(test_df['TransactionDT'].iloc[-1]),
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
}

with open(ARTIFACTS_DIR / 'split_boundaries.json', 'w') as f:
    json.dump(boundaries, f, indent=2)
print('Saved split boundaries:', boundaries)


## 4. Evaluation Function: ROC-AUC, PR-AUC, Precision @ 10% Recall

In [ ]:
def evaluate_model(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    precisions, recalls, _ = precision_recall_curve(y_true, y_prob)
    valid = np.where(recalls >= 0.10)[0]
    p_at_10 = float(np.max(precisions[valid])) if len(valid) > 0 else 0.0
    return {
        'ROC-AUC': round(float(auc), 4),
        'PR-AUC': round(float(ap), 4),
        'Precision@10%Recall': round(p_at_10, 4),
    }


## 5. Baseline A: Weak XGBoost (8 Engineered Features)
Target: AUC ~0.769

In [ ]:
def build_features_a(data):
    X = pd.DataFrame(index=data.index)
    X['TransactionAmt'] = data['TransactionAmt'].fillna(0)
    X['log_TransactionAmt'] = np.log1p(data['TransactionAmt'].clip(lower=0).fillna(0))
    X['ProductCD_code'] = data['ProductCD'].astype('category').cat.codes
    X['card1'] = data['card1'].fillna(-1)
    card4 = data.get('card4', pd.Series(index=data.index))
    X['card4_code'] = card4.astype('category').cat.codes
    X['addr1'] = data['addr1'].fillna(-1)
    X['transaction_hour'] = (data['TransactionDT'] // 3600) % 24
    has_dev = ~data['DeviceInfo'].isna() if 'DeviceInfo' in data else pd.Series(0, index=data.index)
    X['has_device'] = has_dev.astype(int)
    return X

X_train_a = build_features_a(train_df)
X_val_a = build_features_a(val_df)
X_test_a = build_features_a(test_df)
y_train, y_val, y_test = train_df['isFraud'], val_df['isFraud'], test_df['isFraud']

model_a = xgb.XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    scale_pos_weight=float((len(y_train) - sum(y_train)) / sum(y_train)),
    random_state=42, eval_metric='auc', early_stopping_rounds=15, n_jobs=-1,
)
model_a.fit(X_train_a, y_train, eval_set=[(X_val_a, y_val)], verbose=False)
res_a_val = evaluate_model(y_val, model_a.predict_proba(X_val_a)[:, 1])
res_a_test = evaluate_model(y_test, model_a.predict_proba(X_test_a)[:, 1])
print('Baseline A Validation:', res_a_val)
print('Baseline A Test (Target ~0.769):', res_a_test)


## 6. Baseline B: Full Tabular XGBoost (~430 Features)
Target: AUC ~0.907

In [ ]:
drop_cols = ['TransactionID', 'isFraud', 'TransactionDT']
feature_cols = [c for c in df_sorted.columns if c not in drop_cols]

X_train_b = train_df[feature_cols].copy()
X_val_b = val_df[feature_cols].copy()
X_test_b = test_df[feature_cols].copy()

for c in X_train_b.select_dtypes(include=['object', 'category']).columns:
    freq = X_train_b[c].value_counts(normalize=True).to_dict()
    X_train_b[c] = X_train_b[c].map(freq).fillna(0.0).astype(np.float32)
    X_val_b[c] = X_val_b[c].map(freq).fillna(0.0).astype(np.float32)
    X_test_b[c] = X_test_b[c].map(freq).fillna(0.0).astype(np.float32)

model_b = xgb.XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=float((len(y_train) - sum(y_train)) / sum(y_train)),
    random_state=42, eval_metric='auc', early_stopping_rounds=30, n_jobs=-1,
)
model_b.fit(X_train_b, y_train, eval_set=[(X_val_b, y_val)], verbose=False)
res_b_val = evaluate_model(y_val, model_b.predict_proba(X_val_b)[:, 1])
res_b_test = evaluate_model(y_test, model_b.predict_proba(X_test_b)[:, 1])
print('Baseline B Validation:', res_b_val)
print('Baseline B Test (Target ~0.907):', res_b_test)


## 7. Baselines Comparison Table
Compare Baseline A and Baseline B to set the benchmark bar for the Graph Neural Network.

In [ ]:
summary = pd.DataFrame([
    {'Model': 'Baseline A (8 Feats)', 'Val ROC-AUC': res_a_val['ROC-AUC'], 'Test ROC-AUC': res_a_test['ROC-AUC'], 'Test PR-AUC': res_a_test['PR-AUC'], 'Test P@10% Recall': res_a_test['Precision@10%Recall']},
    {'Model': 'Baseline B (All ~430 Feats)', 'Val ROC-AUC': res_b_val['ROC-AUC'], 'Test ROC-AUC': res_b_test['Test ROC-AUC'] if 'Test ROC-AUC' in res_b_test else res_b_test['ROC-AUC'], 'Test PR-AUC': res_b_test['PR-AUC'], 'Test P@10% Recall': res_b_test['Precision@10%Recall']},
])
print(summary)
summary.to_json(ARTIFACTS_DIR / 'baseline_metrics.json', orient='records', indent=2)
